# 6.27 - Approved benchmark plots and tables

Clean reporting notebook for approved benchmark figures and tables. The notebook uses the 10k L1 CertCF run, the random 10k large-support CertCF check with small-support datasets filled from the same CertCF10k rows, the NN-initialized random 10k CertCF alpha 0.25 run, and the approved baseline/FACE result files.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 220,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'grid.linewidth': 0.8,
    'font.size': 10,
})

RESULTS_DIR = ROOT / 'results'
RESULTS_DIR

## Approved Data Load

In [ ]:
RESULT_FILE_SPECS = [
    {
        'result_key': 'final_benchmark_noface',
        'path': RESULTS_DIR / 'final_benchmark_noface.parquet',
        'excluded_methods': {'face'},
        'included_run_names_by_method': {
            # Keep one paper-facing Growing Spheres setting from the sweep.
            'growing_spheres': {'gs_max_radius=50.0'},
        },
    },
    {'result_key': 'face_adult', 'path': RESULTS_DIR / 'benchmark_face_adult.parquet', 'excluded_methods': set()},
    {'result_key': 'face_german_credit', 'path': RESULTS_DIR / 'benchmark_face_german_credit.parquet', 'excluded_methods': set()},
    {'result_key': 'face_compas', 'path': RESULTS_DIR / 'benchmark_face_compas.parquet', 'excluded_methods': set()},
    {'result_key': 'face_wisconsin_breast_cancer', 'path': RESULTS_DIR / 'benchmark_face_wisconsin_breast_cancer.parquet', 'excluded_methods': set()},
    {'result_key': 'face_give_me_some_credit', 'path': RESULTS_DIR / 'benchmark_face_give_me_some_credit.parquet', 'excluded_methods': set()},
    {'result_key': 'face_lending_club', 'path': RESULTS_DIR / 'benchmark_face_lending_club.parquet', 'excluded_methods': set()},
    {'result_key': 'face_heloc', 'path': RESULTS_DIR / 'benchmark_face_heloc.parquet', 'excluded_methods': set()},
]

FILE_STATUS_DF = pd.DataFrame([
    {
        'result_key': spec['result_key'],
        'path': str(spec['path']),
        'available': spec['path'].exists(),
        'size_mb': spec['path'].stat().st_size / (1024 ** 2) if spec['path'].exists() else np.nan,
        'excluded_methods': ', '.join(sorted(spec.get('excluded_methods', set()))),
    }
    for spec in RESULT_FILE_SPECS
])

missing = FILE_STATUS_DF.loc[~FILE_STATUS_DF['available'], 'path'].tolist()
if missing:
    raise FileNotFoundError('Missing approved benchmark result files:\n' + '\n'.join(missing))

frames = []
for spec in RESULT_FILE_SPECS:
    frame = pd.read_parquet(spec['path']).copy()
    excluded_methods = spec.get('excluded_methods', set())
    if excluded_methods:
        frame = frame[~frame['method'].astype(str).isin(excluded_methods)].copy()
    included_run_names_by_method = spec.get('included_run_names_by_method', {})
    for method_name, run_names in included_run_names_by_method.items():
        method_mask = frame['method'].astype(str).eq(str(method_name))
        frame = frame[~method_mask | frame['run_name'].astype(str).isin(run_names)].copy()
    frame['result_key'] = spec['result_key']
    frame['result_file'] = spec['path'].name
    if 'run_name' not in frame.columns:
        frame['run_name'] = frame['method'].astype(str)
    frames.append(frame)

BENCHMARK_DF = pd.concat(frames, ignore_index=True)

print({
    'rows': int(len(BENCHMARK_DF)),
    'datasets': int(BENCHMARK_DF['dataset'].nunique()),
    'methods': sorted(BENCHMARK_DF['method'].dropna().astype(str).unique().tolist()),
    'result_files': int(len(RESULT_FILE_SPECS)),
})

In [ ]:
DATASET_ORDER = [
    'adult',
    'compas',
    'german_credit',
    'give_me_some_credit',
    'heloc',
    'lending_club',
    'wisconsin_breast_cancer',
]

METHOD_ORDER = [
    'CertCF alpha=0.10',
    'CertCF alpha=0.15',
    'CertCF alpha=0.25',
    'CertCF NN-init 10k',
    'CertCF random 10k',
    'NN',
    'NN10000',
    'Growing Spheres',
    'DiCE',
    'FACE',
]

METHOD_LABELS = {
    'nearest_neighbor': 'NN',
    'growing_spheres': 'Growing Spheres',
    'dice': 'DiCE',
    'face': 'FACE',
}

SHORT_METHOD_LABELS = {
    'CertCF alpha=0.10': 'CertCF 0.10',
    'CertCF alpha=0.15': 'CertCF 0.15',
    'CertCF alpha=0.25': 'CertCF',
    'CertCF NN-init 10k': 'CertCF NN-init',
    'CertCF random 10k': 'CertCF random',
    'NN': 'NN',
    'NN10000': 'NN10000',
    'Growing Spheres': 'GS',
    'DiCE': 'DiCE',
    'FACE': 'FACE',
}

PALETTE = {
    'CertCF alpha=0.10': '#099268',
    'CertCF alpha=0.15': '#087F5B',
    'CertCF alpha=0.25': '#0B7285',
    'CertCF NN-init 10k': '#0CA678',
    'CertCF random 10k': '#2F9E44',
    'NN': '#495057',
    'NN10000': '#1864AB',
    'Growing Spheres': '#F08C00',
    'DiCE': '#C92A2A',
    'FACE': '#7048E8',
}

MARKERS = {
    'certcf': 'o',
    'nearest_neighbor': 'X',
    'growing_spheres': '^',
    'dice': 's',
    'face': 'D',
}


def method_label(row: pd.Series) -> str:
    if row['method'] == 'certcf':
        run_name = str(row['run_name'])
        if run_name == 'certcf_random':
            return 'CertCF random 10k'
        if run_name == 'certcf_random_anchor_init_eps_alpha=0.25':
            return 'CertCF NN-init 10k'
        if run_name.startswith('certcf_backward_eps_alpha='):
            return f"CertCF alpha={float(run_name.split('=', 1)[1]):.2f}"
        return run_name.replace('certcf_eps_alpha=', 'CertCF alpha=')
    if row['method'] == 'nearest_neighbor' and str(row['run_name']) in {'nn_k10000', 'nn_probe'}:
        return 'NN10000'
    return METHOD_LABELS.get(str(row['method']), str(row['run_name']))


CERTCF_BASE_LABEL = 'CertCF alpha=0.25'
CERTCF_ANCHOR_INIT_LABEL = 'CertCF NN-init 10k'
CERTCF_RANDOM_LABEL = 'CertCF random 10k'
CERTCF_RANDOM_FILL_RESULT_KEY = 'certcf_random_reduced10000_l1_filled_from_certcf10k'


def add_certcf_random_10k_fill_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['certcf_random_10k_source'] = pd.NA
    out.loc[
        out['method_label'].eq(CERTCF_RANDOM_LABEL),
        'certcf_random_10k_source',
    ] = 'random 10k support run'

    random_datasets = set(
        out.loc[out['method_label'].eq(CERTCF_RANDOM_LABEL), 'dataset']
        .dropna()
        .astype(str)
    )
    fill_datasets = [dataset for dataset in DATASET_ORDER if dataset not in random_datasets]
    fill_rows = out[
        out['dataset'].astype(str).isin(fill_datasets)
        & out['method_label'].eq(CERTCF_BASE_LABEL)
    ].copy()

    if fill_rows.empty:
        return out

    fill_rows['method_label'] = CERTCF_RANDOM_LABEL
    fill_rows['run_name'] = 'certcf_random_filled_from_alpha025'
    fill_rows['result_key'] = CERTCF_RANDOM_FILL_RESULT_KEY
    fill_rows['result_file'] = 'benchmark_full_reduced10000_L1.parquet'
    fill_rows['certcf_random_10k_source'] = 'filled from CertCF10k same support'
    return pd.concat([out, fill_rows], ignore_index=True)


ANALYSIS_DF = BENCHMARK_DF.copy()
ANALYSIS_DF['method_label'] = ANALYSIS_DF.apply(method_label, axis=1)
ANALYSIS_DF = add_certcf_random_10k_fill_rows(ANALYSIS_DF)
ANALYSIS_DF = ANALYSIS_DF[ANALYSIS_DF['method_label'].isin(METHOD_ORDER)].copy()
ANALYSIS_DF['dataset'] = pd.Categorical(ANALYSIS_DF['dataset'], DATASET_ORDER, ordered=True)
ANALYSIS_DF['method_label'] = pd.Categorical(ANALYSIS_DF['method_label'], METHOD_ORDER, ordered=True)
ANALYSIS_DF = ANALYSIS_DF.sort_values(['dataset', 'method_label', 'query_idx'], ignore_index=True)

LOAD_SUMMARY_DF = (
    ANALYSIS_DF
    .groupby(['result_key', 'dataset', 'method', 'run_name'], observed=True, dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['dataset', 'method', 'run_name', 'result_key'], ignore_index=True)
)

LOAD_SUMMARY_DF.head()

In [ ]:
def summarize_validity_l1(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (dataset, method_label), group in df.groupby(['dataset', 'method_label'], observed=True, dropna=False):
        success_group = group[group['success']]
        rows.append({
            'dataset': str(dataset),
            'method_label': str(method_label),
            'method': str(group['method'].iloc[0]),
            'run_name': str(group['run_name'].iloc[0]),
            'n': int(len(group)),
            'n_valid': int(group['success'].sum()),
            'validity_pct': float(100.0 * group['success'].mean()),
            'l1_mean': float(success_group['l1_distance'].mean()) if len(success_group) else np.nan,
            'l1_median': float(success_group['l1_distance'].median()) if len(success_group) else np.nan,
            'mad_l1_mean': float(success_group['mad_l1_distance'].mean()) if len(success_group) else np.nan,
            'runtime_mean_s': float(group['runtime_s'].mean()),
        })
    out = pd.DataFrame(rows)
    out['dataset'] = pd.Categorical(out['dataset'], DATASET_ORDER, ordered=True)
    out['method_label'] = pd.Categorical(out['method_label'], METHOD_ORDER, ordered=True)
    return out.sort_values(['dataset', 'method_label'], ignore_index=True)


VALIDITY_L1_DF = summarize_validity_l1(ANALYSIS_DF)
VALIDITY_L1_DF.shape

## Large-Support 10k Random Check

Aggregate tables and plots treat `CertCF random 10k` as a seven-dataset method: the three large-support datasets use the new random run, while the remaining datasets are filled from the existing CertCF10k rows where the support is unchanged. `CertCF NN-init 10k` is loaded from the all-dataset random-anchor initialization sweep at alpha 0.25. This table isolates the large-support random comparisons.


In [ ]:
LARGE_SUPPORT_DATASETS = [
    'adult',
    'give_me_some_credit',
    'lending_club',
]

LARGE_SUPPORT_LABELS = [
    'CertCF alpha=0.10',
    'CertCF alpha=0.15',
    'NN10000',
]


def build_large_support_random_comparison(df: pd.DataFrame) -> pd.DataFrame:
    subset = df[
        df['dataset'].astype(str).isin(LARGE_SUPPORT_DATASETS)
        & df['method_label'].astype(str).isin(LARGE_SUPPORT_LABELS)
    ].copy()

    rows = []
    for dataset in LARGE_SUPPORT_DATASETS:
        ds = subset[subset['dataset'].astype(str).eq(dataset)].copy()
        wide = ds.pivot_table(
            index=['dataset', 'query_idx', 'target_class'],
            columns='method_label',
            values=['success', 'l1_distance'],
            aggfunc='first',
            observed=True,
        )
        if wide.empty:
            continue

        present = [label for label in LARGE_SUPPORT_LABELS if ('l1_distance', label) in wide.columns]
        required = ['CertCF alpha=0.15', 'NN10000']
        if not all(label in present for label in required):
            continue

        common_mask = np.ones(len(wide), dtype=bool)
        for label in present:
            common_mask &= wide[('l1_distance', label)].notna().to_numpy()
        common = wide.loc[common_mask]
        if common.empty:
            continue

        certcf_l1 = common[('l1_distance', 'CertCF alpha=0.15')].astype(float)
        nn_l1 = common[('l1_distance', 'NN10000')].astype(float)

        row = {
            'dataset': dataset,
            'n_common': int(len(common)),
            'certcf_alpha015_validity_pct': float(100.0 * common[('success', 'CertCF alpha=0.15')].astype(bool).mean()),
            'nn10000_validity_pct': float(100.0 * common[('success', 'NN10000')].astype(bool).mean()),
            'certcf_alpha015_l1_mean': float(certcf_l1.mean()),
            'nn10000_l1_mean': float(nn_l1.mean()),
            'certcf_alpha015_minus_nn10000_l1_mean': float((certcf_l1 - nn_l1).mean()),
            'certcf_alpha015_le_nn10000_pct': float(100.0 * (certcf_l1 <= nn_l1).mean()),
        }

        if 'CertCF alpha=0.10' in present:
            alpha_l1 = common[('l1_distance', 'CertCF alpha=0.10')].astype(float)
            row.update({
                'certcf_alpha010_validity_pct': float(100.0 * common[('success', 'CertCF alpha=0.10')].astype(bool).mean()),
                'certcf_alpha010_l1_mean': float(alpha_l1.mean()),
                'alpha015_minus_alpha010_l1_mean': float((certcf_l1 - alpha_l1).mean()),
                'alpha015_le_alpha010_pct': float(100.0 * (certcf_l1 <= alpha_l1).mean()),
            })

        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=['dataset', 'n_common'])
    out['dataset'] = pd.Categorical(out['dataset'], LARGE_SUPPORT_DATASETS, ordered=True)
    return out.sort_values('dataset', ignore_index=True)


LARGE_SUPPORT_RANDOM_COMPARISON_DF = build_large_support_random_comparison(ANALYSIS_DF)
display(LARGE_SUPPORT_RANDOM_COMPARISON_DF.round(4))


## Validity vs L1 Proximity

In [ ]:
def build_validity_l1_curve_df(df: pd.DataFrame, n_thresholds: int = 350) -> pd.DataFrame:
    rows = []

    for dataset in DATASET_ORDER:
        ds_df = df[df['dataset'].astype(str).eq(dataset)].copy()
        success_l1 = ds_df.loc[ds_df['success'], 'l1_distance'].dropna().to_numpy(dtype=float)
        if len(success_l1) == 0:
            continue

        thresholds = np.linspace(0.0, float(success_l1.max()), n_thresholds)
        for method_label in METHOD_ORDER:
            method_df = ds_df[ds_df['method_label'].astype(str).eq(method_label)].copy()
            if method_df.empty:
                continue

            distances = method_df['l1_distance'].to_numpy(dtype=float)
            distances = np.where(method_df['success'].to_numpy(dtype=bool), distances, np.inf)
            finite_distances = np.sort(distances[np.isfinite(distances)])
            cumulative_validity = np.searchsorted(finite_distances, thresholds, side='right') / len(method_df)
            final_validity_pct = 100.0 * float(method_df['success'].astype(bool).mean())

            for threshold, validity in zip(thresholds, cumulative_validity):
                rows.append({
                    'dataset': dataset,
                    'method_label': method_label,
                    'method': str(method_df['method'].iloc[0]),
                    'l1_threshold': float(threshold),
                    'validity_pct': float(100.0 * validity),
                    'final_validity_pct': float(final_validity_pct),
                })

    return pd.DataFrame(rows)


def format_validity_label(value: float) -> str:
    return '100%' if np.isclose(value, 100.0) else f'{value:.1f}%'


def spread_endpoint_label_positions(
    endpoints: list[dict],
    min_gap: float = 4.0,
    y_min: float = 3.0,
    y_max: float = 99.5,
) -> np.ndarray:
    if not endpoints:
        return np.array([], dtype=float)

    y = np.array([endpoint.get('final_y', endpoint['y']) for endpoint in endpoints], dtype=float)
    placed = np.clip(y, y_min, y_max)
    order = np.argsort(placed)

    for prev_idx, idx in zip(order[:-1], order[1:]):
        if placed[idx] < placed[prev_idx] + min_gap:
            placed[idx] = placed[prev_idx] + min_gap

    overflow = placed[order[-1]] - y_max
    if overflow > 0:
        placed[order] -= overflow

    for next_idx, idx in zip(order[:0:-1], order[-2::-1]):
        if placed[idx] > placed[next_idx] - min_gap:
            placed[idx] = placed[next_idx] - min_gap

    return np.clip(placed, y_min, y_max)


def add_endpoint_validity_labels(
    ax,
    endpoints: list[dict],
    *,
    x_axes: float = 1.01,
    fontsize: float = 7.5,
    min_gap: float = 4.0,
    include_method: bool = True,
) -> None:
    label_y = spread_endpoint_label_positions(endpoints, min_gap=min_gap)
    for endpoint, y_label in zip(endpoints, label_y):
        method_label = endpoint['method_label']
        short = SHORT_METHOD_LABELS.get(method_label, method_label)
        final_y = float(endpoint.get('final_y', endpoint['y']))
        text = f"{short} {format_validity_label(final_y)}" if include_method else format_validity_label(final_y)
        color = endpoint['color']
        ax.annotate(
            text,
            xy=(1.0, endpoint['y']),
            xycoords=('axes fraction', 'data'),
            xytext=(x_axes, float(y_label)),
            textcoords=('axes fraction', 'data'),
            ha='left',
            va='center',
            color=color,
            fontsize=fontsize,
            fontweight='bold' if method_label in {CERTCF_BASE_LABEL, CERTCF_ANCHOR_INIT_LABEL} else 'normal',
            clip_on=False,
            annotation_clip=False,
            arrowprops={
                'arrowstyle': '-',
                'color': color,
                'linewidth': 0.75,
                'alpha': 0.55,
                'shrinkA': 0,
                'shrinkB': 0,
            },
        )


def plot_validity_vs_l1_ecdf(curve_df: pd.DataFrame):
    fig, axes = plt.subplots(4, 2, figsize=(13.5, 15.0), sharey=True)
    axes = axes.ravel()

    for ax, dataset in zip(axes, DATASET_ORDER):
        ds_curve = curve_df[curve_df['dataset'].eq(dataset)].copy()
        if ds_curve.empty:
            ax.set_visible(False)
            continue

        endpoints = []
        for method_label in METHOD_ORDER:
            line_df = ds_curve[ds_curve['method_label'].eq(method_label)]
            if line_df.empty:
                continue
            method = str(line_df['method'].iloc[0])
            ax.step(
                line_df['l1_threshold'],
                line_df['validity_pct'],
                where='post',
                color=PALETTE.get(method_label, '#868E96'),
                linewidth=2.2 if method == 'certcf' else 1.9,
                alpha=0.95 if method == 'certcf' else 0.82,
                label=method_label,
            )
            endpoints.append({
                'method_label': method_label,
                'y': float(line_df['validity_pct'].iloc[-1]),
                'final_y': float(line_df['final_validity_pct'].iloc[0]),
                'color': PALETTE.get(method_label, '#868E96'),
            })

        ax.axhline(100.0, color='#2B8A3E', linewidth=1.0, linestyle='--', alpha=0.75)
        ax.axhline(99.0, color='#ADB5BD', linewidth=0.9, linestyle=':', alpha=0.75)
        ax.set_ylim(-2, 102.5)
        ax.set_xlim(left=0.0)
        ax.set_title(dataset.replace('_', ' '), fontsize=11, fontweight='bold')
        ax.set_xlabel('L1 proximity threshold')
        ax.set_ylabel('Valid CFs within threshold (%)')
        ax.grid(True, axis='both', alpha=0.22)
        add_endpoint_validity_labels(ax, endpoints, fontsize=7.0, min_gap=3.8)

    for ax in axes[len(DATASET_ORDER):]:
        ax.set_visible(False)

    present_methods = set(curve_df['method_label'].astype(str))
    legend_handles = [
        plt.Line2D([], [], color=PALETTE.get(label, '#868E96'), linewidth=2.4, label=label)
        for label in METHOD_ORDER
        if label in present_methods
    ]
    fig.legend(
        legend_handles,
        [handle.get_label() for handle in legend_handles],
        loc='lower center',
        ncol=5,
        frameon=False,
        bbox_to_anchor=(0.5, 0.01),
    )
    fig.suptitle('Validity vs L1 proximity', fontsize=16, fontweight='bold', y=0.995)
    fig.tight_layout(rect=(0, 0.055, 0.92, 0.97), w_pad=2.8)
    return fig


VALIDITY_L1_CURVE_DF = build_validity_l1_curve_df(ANALYSIS_DF)
plot_validity_vs_l1_ecdf(VALIDITY_L1_CURVE_DF);

## Dataset-Balanced Normalized Validity vs Proximity

In [ ]:
NORMALIZED_L1_X_MAX = 2.5
NORMALIZED_L1_N_THRESHOLDS = 350


def nn_l1_scale_by_dataset(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset in DATASET_ORDER:
        ds_df = df[df['dataset'].astype(str).eq(dataset)].copy()
        nn_success = ds_df[
            ds_df['method_label'].astype(str).eq('NN')
            & ds_df['success'].astype(bool)
            & ds_df['l1_distance'].notna()
        ]
        all_success = ds_df[ds_df['success'].astype(bool) & ds_df['l1_distance'].notna()]

        if not nn_success.empty:
            scale = float(nn_success['l1_distance'].mean())
            scale_source = 'NN mean successful L1 (full support)'
        elif not all_success.empty:
            scale = float(all_success['l1_distance'].median())
            scale_source = 'all-method median successful L1 fallback'
        else:
            scale = np.nan
            scale_source = 'missing'

        rows.append({
            'dataset': dataset,
            'l1_scale': scale,
            'scale_source': scale_source,
            'nn_success_count': int(len(nn_success)),
        })

    return pd.DataFrame(rows)


def build_dataset_balanced_normalized_l1_curve_df(
    df: pd.DataFrame,
    scale_df: pd.DataFrame,
    x_max: float = NORMALIZED_L1_X_MAX,
    n_thresholds: int = NORMALIZED_L1_N_THRESHOLDS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    thresholds = np.linspace(0.0, x_max, n_thresholds)
    scale_by_dataset = scale_df.set_index('dataset')['l1_scale'].to_dict()
    dataset_curve_rows = []

    for dataset in DATASET_ORDER:
        scale = scale_by_dataset.get(dataset, np.nan)
        if not np.isfinite(scale) or scale <= 0:
            continue

        ds_df = df[df['dataset'].astype(str).eq(dataset)].copy()
        for method_label in METHOD_ORDER:
            method_df = ds_df[ds_df['method_label'].astype(str).eq(method_label)].copy()
            if method_df.empty:
                continue

            final_validity_pct = 100.0 * float(method_df['success'].astype(bool).mean())
            distances = method_df['l1_distance'].to_numpy(dtype=float) / scale
            distances = np.where(method_df['success'].to_numpy(dtype=bool), distances, np.inf)
            finite_distances = np.sort(distances[np.isfinite(distances)])
            cumulative_validity = np.searchsorted(finite_distances, thresholds, side='right') / len(method_df)

            for threshold, validity in zip(thresholds, cumulative_validity):
                dataset_curve_rows.append({
                    'dataset': dataset,
                    'method_label': method_label,
                    'method': str(method_df['method'].iloc[0]),
                    'normalized_l1_threshold': float(threshold),
                    'validity_pct': float(100.0 * validity),
                    'final_validity_pct': float(final_validity_pct),
                })

    dataset_curve_df = pd.DataFrame(dataset_curve_rows)
    aggregate_curve_df = (
        dataset_curve_df
        .groupby(['method_label', 'method', 'normalized_l1_threshold'], observed=True, sort=False)
        .agg(
            mean_validity_pct=('validity_pct', 'mean'),
            std_validity_pct=('validity_pct', 'std'),
            mean_final_validity_pct=('final_validity_pct', 'mean'),
            n_datasets=('dataset', 'nunique'),
        )
        .reset_index()
    )
    aggregate_curve_df['std_validity_pct'] = aggregate_curve_df['std_validity_pct'].fillna(0.0)
    return dataset_curve_df, aggregate_curve_df


def normalized_curve_auc_table(aggregate_curve_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method_label in METHOD_ORDER:
        curve = aggregate_curve_df[aggregate_curve_df['method_label'].eq(method_label)].sort_values('normalized_l1_threshold')
        if curve.empty:
            continue
        x = curve['normalized_l1_threshold'].to_numpy(dtype=float)
        y = curve['mean_validity_pct'].to_numpy(dtype=float) / 100.0
        area = np.trapezoid(y, x) if hasattr(np, 'trapezoid') else np.trapz(y, x)
        rows.append({
            'method_label': method_label,
            'n_datasets': int(curve['n_datasets'].iloc[0]),
            'auc_0_to_2_5': float(area / NORMALIZED_L1_X_MAX),
            'validity_at_1x_nn_l1': float(np.interp(1.0, x, curve['mean_validity_pct'].to_numpy(dtype=float))),
            'validity_at_2x_nn_l1': float(np.interp(2.0, x, curve['mean_validity_pct'].to_numpy(dtype=float))),
        })
    return pd.DataFrame(rows)


def plot_dataset_balanced_normalized_l1_curve(aggregate_curve_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(10.5, 6.2))

    endpoints = []
    for method_label in METHOD_ORDER:
        curve = aggregate_curve_df[aggregate_curve_df['method_label'].eq(method_label)].sort_values('normalized_l1_threshold')
        if curve.empty:
            continue
        x = curve['normalized_l1_threshold'].to_numpy(dtype=float)
        y = curve['mean_validity_pct'].to_numpy(dtype=float)
        y_std = curve['std_validity_pct'].to_numpy(dtype=float)
        y_low = np.clip(y - y_std, 0.0, 100.0)
        y_high = np.clip(y + y_std, 0.0, 100.0)
        method = str(curve['method'].iloc[0])
        color = PALETTE.get(method_label, '#868E96')
        ax.step(
            x,
            y,
            where='post',
            color=color,
            linewidth=2.7 if method == 'certcf' else 2.2,
            alpha=0.96 if method == 'certcf' else 0.82,
            label=method_label,
        )
        ax.fill_between(
            x,
            y_low,
            y_high,
            step='post',
            color=color,
            alpha=0.12 if method == 'certcf' else 0.08,
            linewidth=0,
        )
        for boundary in (y_low, y_high):
            ax.step(
                x,
                boundary,
                where='post',
                color=color,
                linewidth=0.85 if method == 'certcf' else 0.75,
                alpha=0.36 if method == 'certcf' else 0.26,
                zorder=1.8,
            )
        endpoints.append({
            'method_label': method_label,
            'y': float(y[-1]),
            'final_y': float(curve['mean_final_validity_pct'].iloc[-1]),
            'color': color,
        })

    ax.axhline(100.0, color='#2B8A3E', linewidth=1.0, linestyle='--', alpha=0.65)
    ax.axhline(99.0, color='#ADB5BD', linewidth=0.9, linestyle=':', alpha=0.75)
    ax.set_xlim(0.0, NORMALIZED_L1_X_MAX)
    ax.set_ylim(-2.0, 102.5)
    ax.set_xlabel('Normalized L1 proximity threshold (L1 / dataset NN mean L1)')
    ax.set_ylabel('Dataset-averaged valid CFs within threshold (%)')
    ax.set_title('Dataset-averaged validity vs normalized L1 proximity', fontsize=14, fontweight='bold')
    ax.grid(True, axis='both', alpha=0.22)
    add_endpoint_validity_labels(ax, endpoints, fontsize=9.0, min_gap=4.6)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0.08, 0.88, 1))
    return fig


NORMALIZED_L1_SCALE_DF = nn_l1_scale_by_dataset(ANALYSIS_DF)
DATASET_NORMALIZED_L1_CURVE_DF, AGGREGATE_NORMALIZED_L1_CURVE_DF = build_dataset_balanced_normalized_l1_curve_df(
    ANALYSIS_DF,
    NORMALIZED_L1_SCALE_DF,
)
NORMALIZED_L1_AUC_DF = normalized_curve_auc_table(AGGREGATE_NORMALIZED_L1_CURVE_DF)

plot_dataset_balanced_normalized_l1_curve(AGGREGATE_NORMALIZED_L1_CURVE_DF);
display(NORMALIZED_L1_AUC_DF.round(4))

## Mixed Empirical Robustness

Empirical robustness under valid mixed tabular perturbations: numerical dimensions receive random Lp noise, while categorical features are perturbed by flipping entire one-hot blocks to another valid category. Immutable categorical features declared in the dataset spec are kept fixed. The strict score counts a counterfactual as robust only if all sampled perturbations preserve the requested target class; the preservation score averages target preservation over all sampled perturbations.

In [ ]:
from notebooks.utils.robustness import load_tabular_benchmark_models, sample_lp_ball
from dataset_specs import get_tabular_dataset_spec

MIXED_ROBUSTNESS_CONFIG_PATH = ROOT / 'configs' / 'benchmarks' / 'full_reduced10000_L1.yaml'
MIXED_ROBUSTNESS_DEVICE = 'cpu'
MIXED_ROBUSTNESS_SEED = 42
MIXED_ROBUSTNESS_N_SAMPLES = 32
MIXED_ROBUSTNESS_NUMERIC_NORM = 'l2'
MIXED_ROBUSTNESS_NUMERIC_EPS = [0.0, 0.05, 0.10, 0.20, 0.30]
MIXED_ROBUSTNESS_CAT_FLIP_PROBS = [0.0, 0.02, 0.05, 0.10]
MIXED_ROBUSTNESS_METHOD_ORDER = [
    'CertCF NN-init 10k',
    'NN10000',
    'Growing Spheres',
    'FACE',
    'DiCE',
]
MIXED_ROBUSTNESS_SCORE_PATH = (
    RESULTS_DIR
    / 'approved_plot_data'
    / f'mixed_empirical_robustness_approved_{MIXED_ROBUSTNESS_NUMERIC_NORM}'
      f'_s{MIXED_ROBUSTNESS_N_SAMPLES}_seed{MIXED_ROBUSTNESS_SEED}_base_consistent.parquet'
)
RECOMPUTE_MIXED_ROBUSTNESS = False


def mixed_robustness_feature_columns(d: int) -> list[str]:
    return [f'x_cf_{idx}' for idx in range(int(d))]


def mixed_robustness_numerical_indices(dataset_name: str) -> np.ndarray:
    spec = get_tabular_dataset_spec(dataset_name)
    return np.asarray(
        [idx for idx, feature_type in enumerate(spec.ohe_feature_types) if feature_type == 'numerical'],
        dtype=np.int64,
    )


def mixed_robustness_categorical_blocks(
    dataset_name: str,
    *,
    include_immutable: bool = False,
) -> list[tuple[str, int, int]]:
    spec = get_tabular_dataset_spec(dataset_name)
    blocks = []
    for feature_name, feature_type, (start, end) in zip(
        spec.feature_names,
        spec.input_types,
        spec.feature_slices,
    ):
        if feature_type != 'categorical':
            continue
        if not include_immutable and feature_name in spec.immutable_features:
            continue
        blocks.append((feature_name, int(start), int(end)))
    return blocks


def flip_ohe_blocks(
    rng: np.random.Generator,
    x: np.ndarray,
    blocks: list[tuple[str, int, int]],
    flip_prob: float,
) -> np.ndarray:
    """Randomly flip whole OHE blocks to another valid category."""
    out = np.asarray(x, dtype=np.float32).copy()
    flip_prob = float(flip_prob)
    if out.ndim != 2:
        raise ValueError('x must be a 2-D array.')
    if flip_prob <= 0.0 or not blocks:
        return out

    flip_prob = min(max(flip_prob, 0.0), 1.0)
    n_rows = out.shape[0]
    row_idx_all = np.arange(n_rows)
    for _, start, end in blocks:
        width = int(end - start)
        if width <= 1:
            continue
        mask = rng.random(n_rows) < flip_prob
        if not mask.any():
            continue
        row_idx = row_idx_all[mask]
        current = np.argmax(out[row_idx, start:end], axis=1)
        offsets = rng.integers(1, width, size=len(row_idx))
        new_category = (current + offsets) % width
        out[row_idx, start:end] = 0.0
        out[row_idx, start + new_category] = 1.0
    return out


def sample_mixed_tabular_perturbations(
    rng: np.random.Generator,
    x_cf: np.ndarray,
    *,
    dataset_name: str,
    n_samples: int,
    numeric_epsilon: float,
    numeric_norm: str = MIXED_ROBUSTNESS_NUMERIC_NORM,
    categorical_flip_prob: float = 0.0,
    include_immutable_categorical: bool = False,
) -> np.ndarray:
    """Sample valid mixed perturbations around one counterfactual."""
    spec = get_tabular_dataset_spec(dataset_name)
    base = np.repeat(np.asarray(x_cf, dtype=np.float32)[None, :], int(n_samples), axis=0)
    if base.shape[1] != spec.n_features:
        raise ValueError(
            f'Dataset {dataset_name!r} expected {spec.n_features} features, got {base.shape[1]}.'
        )

    numeric_idx = mixed_robustness_numerical_indices(dataset_name)
    if numeric_idx.size > 0 and numeric_epsilon > 0:
        noise = sample_lp_ball(
            rng,
            n_samples=int(n_samples),
            dim=int(numeric_idx.size),
            radius=float(numeric_epsilon),
            norm=numeric_norm,
        )
        base[:, numeric_idx] += noise

    blocks = mixed_robustness_categorical_blocks(
        dataset_name,
        include_immutable=include_immutable_categorical,
    )
    return flip_ohe_blocks(rng, base, blocks, categorical_flip_prob)


def evaluate_mixed_empirical_robustness_grid(
    df: pd.DataFrame,
    *,
    models_by_dataset: dict[str, object],
    dataset_order: list[str],
    method_order: list[str],
    numeric_eps: list[float],
    categorical_flip_probs: list[float],
    numeric_norm: str = MIXED_ROBUSTNESS_NUMERIC_NORM,
    n_samples: int = MIXED_ROBUSTNESS_N_SAMPLES,
    seed: int = MIXED_ROBUSTNESS_SEED,
    target_col: str = 'target_class',
    include_immutable_categorical: bool = False,
) -> pd.DataFrame:
    data = df[df['success'].astype(bool)].copy()
    if data.empty:
        return pd.DataFrame()

    if target_col not in data.columns:
        source_col = 'source_class' if 'source_class' in data.columns else 'y_orig' if 'y_orig' in data.columns else None
        if source_col is None:
            raise ValueError(f'Missing {target_col!r} and no source_class/y_orig fallback is available.')
        source_values = pd.to_numeric(data[source_col], errors='coerce')
        unique_values = set(source_values.dropna().astype(int).unique().tolist())
        if not unique_values.issubset({0, 1}):
            raise ValueError('Binary source-class fallback is only valid for binary tasks.')
        data[target_col] = 1 - source_values.astype(np.int64)

    rng = np.random.default_rng(seed)
    rows = []
    for dataset_name in dataset_order:
        if dataset_name not in models_by_dataset:
            continue
        spec = get_tabular_dataset_spec(dataset_name)
        ds_df = data[data['dataset'].astype(str).eq(dataset_name)].copy()
        if ds_df.empty:
            continue
        cf_cols = mixed_robustness_feature_columns(spec.n_features)
        if not set(cf_cols).issubset(ds_df.columns):
            continue

        numeric_idx = mixed_robustness_numerical_indices(dataset_name)
        cat_blocks = mixed_robustness_categorical_blocks(
            dataset_name,
            include_immutable=include_immutable_categorical,
        )
        model = models_by_dataset[dataset_name]

        for method_label in method_order:
            method_df = ds_df[ds_df['method_label'].astype(str).eq(method_label)].copy()
            method_df = method_df[method_df[target_col].notna()].copy()
            if method_df.empty:
                continue

            x_cf_all = method_df[cf_cols].to_numpy(dtype=np.float32)
            targets_all = method_df[target_col].astype(int).to_numpy(dtype=np.int64)
            finite = np.isfinite(x_cf_all).all(axis=1)
            x_cf_all = x_cf_all[finite]
            targets_all = targets_all[finite]
            if len(x_cf_all) == 0:
                continue

            base_preds = model.predict(x_cf_all)
            base_consistent = base_preds == targets_all
            n_benchmark_success = int(len(x_cf_all))
            n_base_consistent = int(base_consistent.sum())
            base_consistency_pct = 100.0 * n_base_consistent / n_benchmark_success
            x_cf = x_cf_all[base_consistent]
            targets = targets_all[base_consistent]
            if len(x_cf) == 0:
                continue

            for numeric_epsilon in numeric_eps:
                for categorical_flip_prob in categorical_flip_probs:
                    perturbed = np.concatenate(
                        [
                            sample_mixed_tabular_perturbations(
                                rng,
                                row,
                                dataset_name=dataset_name,
                                n_samples=n_samples,
                                numeric_epsilon=float(numeric_epsilon),
                                numeric_norm=numeric_norm,
                                categorical_flip_prob=float(categorical_flip_prob),
                                include_immutable_categorical=include_immutable_categorical,
                            )
                            for row in x_cf
                        ],
                        axis=0,
                    )
                    preds = model.predict(perturbed).reshape(len(x_cf), n_samples)
                    preserved = preds == targets[:, None]
                    robust_flags = preserved.all(axis=1)

                    rows.append({
                        'dataset': dataset_name,
                        'method_label': method_label,
                        'numeric_norm': numeric_norm,
                        'numeric_epsilon': float(numeric_epsilon),
                        'categorical_flip_prob': float(categorical_flip_prob),
                        'success_pct': float(100.0 * robust_flags.mean()),
                        'mean_preservation_pct': float(100.0 * preserved.mean()),
                        'n_queries': int(len(x_cf)),
                        'n_benchmark_success': n_benchmark_success,
                        'n_base_consistent': n_base_consistent,
                        'base_consistency_pct': float(base_consistency_pct),
                        'n_samples': int(n_samples),
                        'n_numeric_dims': int(numeric_idx.size),
                        'n_mutable_categorical_blocks': int(len(cat_blocks)),
                    })

    return pd.DataFrame(rows)


def aggregate_mixed_robustness(curve_df: pd.DataFrame) -> pd.DataFrame:
    if curve_df.empty:
        return pd.DataFrame()
    return (
        curve_df
        .groupby(['method_label', 'numeric_norm', 'numeric_epsilon', 'categorical_flip_prob'], observed=True, sort=False)
        .agg(
            mean_success_pct=('success_pct', 'mean'),
            std_success_pct=('success_pct', 'std'),
            mean_preservation_pct=('mean_preservation_pct', 'mean'),
            std_preservation_pct=('mean_preservation_pct', 'std'),
            n_datasets=('dataset', 'nunique'),
            total_queries=('n_queries', 'sum'),
            total_benchmark_success=('n_benchmark_success', 'sum'),
            total_base_consistent=('n_base_consistent', 'sum'),
            mean_base_consistency_pct=('base_consistency_pct', 'mean'),
        )
        .reset_index()
        .fillna({'std_success_pct': 0.0, 'std_preservation_pct': 0.0})
    )


def plot_mixed_robustness_numeric_curves(
    aggregate_df: pd.DataFrame,
    *,
    categorical_flip_prob: float = 0.05,
    y_col: str = 'mean_success_pct',
):
    fig, ax = plt.subplots(figsize=(10.2, 5.8))
    subset = aggregate_df[np.isclose(aggregate_df['categorical_flip_prob'].astype(float), categorical_flip_prob)].copy()
    for method_label in MIXED_ROBUSTNESS_METHOD_ORDER:
        curve = subset[subset['method_label'].astype(str).eq(method_label)].sort_values('numeric_epsilon')
        if curve.empty:
            continue
        color = PALETTE.get(method_label, '#868E96')
        ax.plot(
            curve['numeric_epsilon'],
            curve[y_col],
            marker='o',
            linewidth=2.4 if method_label.startswith('CertCF') else 2.0,
            color=color,
            label=method_label,
            alpha=0.94,
        )
    ax.set_xlabel(f'Numerical perturbation radius ({MIXED_ROBUSTNESS_NUMERIC_NORM.upper()})')
    ylabel = 'Strict robust CFs (%)' if y_col == 'mean_success_pct' else 'Mean target preservation (%)'
    ax.set_ylabel(ylabel)
    ax.set_title(
        f'Mixed empirical robustness, categorical flip p={categorical_flip_prob:g}',
        fontsize=14,
        fontweight='bold',
    )
    ax.set_ylim(-2.0, 102.0)
    ax.grid(True, axis='both', alpha=0.22)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    return fig


def plot_mixed_robustness_categorical_bars(
    aggregate_df: pd.DataFrame,
    *,
    numeric_epsilon: float = 0.10,
    categorical_flip_prob: float = 0.05,
    y_col: str = 'mean_success_pct',
):
    subset = aggregate_df[
        np.isclose(aggregate_df['numeric_epsilon'].astype(float), numeric_epsilon)
        & np.isclose(aggregate_df['categorical_flip_prob'].astype(float), categorical_flip_prob)
    ].copy()
    subset['method_label'] = pd.Categorical(subset['method_label'], MIXED_ROBUSTNESS_METHOD_ORDER, ordered=True)
    subset = subset.sort_values('method_label')

    fig, ax = plt.subplots(figsize=(9.2, 4.8))
    colors = [PALETTE.get(str(label), '#868E96') for label in subset['method_label']]
    ax.bar(subset['method_label'].astype(str), subset[y_col], color=colors, alpha=0.88)
    ylabel = 'Strict robust CFs (%)' if y_col == 'mean_success_pct' else 'Mean target preservation (%)'
    ax.set_ylabel(ylabel)
    ax.set_title(
        f'Mixed robustness at epsilon={numeric_epsilon:g}, categorical flip p={categorical_flip_prob:g}',
        fontsize=13,
        fontweight='bold',
    )
    ax.set_ylim(0.0, 102.0)
    ax.tick_params(axis='x', rotation=25)
    ax.grid(True, axis='y', alpha=0.22)
    fig.tight_layout()
    return fig


if MIXED_ROBUSTNESS_SCORE_PATH.exists() and not RECOMPUTE_MIXED_ROBUSTNESS:
    MIXED_ROBUSTNESS_CURVE_DF = pd.read_parquet(MIXED_ROBUSTNESS_SCORE_PATH)
else:
    MIXED_ROBUSTNESS_SCORE_PATH.parent.mkdir(parents=True, exist_ok=True)
    MIXED_ROBUSTNESS_MODELS = load_tabular_benchmark_models(
        MIXED_ROBUSTNESS_CONFIG_PATH,
        datasets=DATASET_ORDER,
        device=MIXED_ROBUSTNESS_DEVICE,
    )
    MIXED_ROBUSTNESS_CURVE_DF = evaluate_mixed_empirical_robustness_grid(
        ANALYSIS_DF,
        models_by_dataset=MIXED_ROBUSTNESS_MODELS,
        dataset_order=DATASET_ORDER,
        method_order=MIXED_ROBUSTNESS_METHOD_ORDER,
        numeric_eps=MIXED_ROBUSTNESS_NUMERIC_EPS,
        categorical_flip_probs=MIXED_ROBUSTNESS_CAT_FLIP_PROBS,
        numeric_norm=MIXED_ROBUSTNESS_NUMERIC_NORM,
        n_samples=MIXED_ROBUSTNESS_N_SAMPLES,
        seed=MIXED_ROBUSTNESS_SEED,
    )
    MIXED_ROBUSTNESS_CURVE_DF.to_parquet(MIXED_ROBUSTNESS_SCORE_PATH, index=False, compression='gzip')

MIXED_ROBUSTNESS_AGG_DF = aggregate_mixed_robustness(MIXED_ROBUSTNESS_CURVE_DF)
MIXED_ROBUSTNESS_AGG_DF['method_label'] = pd.Categorical(
    MIXED_ROBUSTNESS_AGG_DF['method_label'], MIXED_ROBUSTNESS_METHOD_ORDER, ordered=True
)
MIXED_ROBUSTNESS_AGG_DF = MIXED_ROBUSTNESS_AGG_DF.sort_values(
    ['categorical_flip_prob', 'numeric_epsilon', 'method_label'],
    ignore_index=True,
)

MIXED_ROBUSTNESS_ZERO_CHECK_DF = MIXED_ROBUSTNESS_AGG_DF[
    np.isclose(MIXED_ROBUSTNESS_AGG_DF['numeric_epsilon'].astype(float), 0.0)
    & np.isclose(MIXED_ROBUSTNESS_AGG_DF['categorical_flip_prob'].astype(float), 0.0)
][[
    'method_label',
    'mean_success_pct',
    'mean_preservation_pct',
    'total_benchmark_success',
    'total_base_consistent',
    'mean_base_consistency_pct',
]].copy()

print({
    'mixed_robustness_rows': int(len(MIXED_ROBUSTNESS_CURVE_DF)),
    'cache': str(MIXED_ROBUSTNESS_SCORE_PATH),
    'n_samples': MIXED_ROBUSTNESS_N_SAMPLES,
})
display(MIXED_ROBUSTNESS_ZERO_CHECK_DF.round(4))
display(MIXED_ROBUSTNESS_AGG_DF.round(4))
plot_mixed_robustness_numeric_curves(MIXED_ROBUSTNESS_AGG_DF, categorical_flip_prob=0.05);
plot_mixed_robustness_categorical_bars(MIXED_ROBUSTNESS_AGG_DF, numeric_epsilon=0.10, categorical_flip_prob=0.05);

## LOF Plausibility

Local Outlier Factor is fitted once per dataset on the training set and evaluated on successful counterfactuals. We compute the raw outlier score as `-score_samples`, so lower values are less outlier-like. Raw LOF means can be dominated by a few extreme novelty scores, so the paper-facing summaries emphasize the median raw LOF and the mean of `log(1 + raw LOF)`, averaged across datasets.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
from counterfactuals.benchmarks import create_default_registries

LOF_N_NEIGHBORS = 20
LOF_SCORE_PATH = RESULTS_DIR / 'approved_plot_data' / 'lof_scores_approved_with_certcf_base.parquet'
RECOMPUTE_LOF = False
PAPER_LOF_METHOD_ORDER = [
    'CertCF alpha=0.25',
    'CertCF NN-init 10k',
    'NN10000',
    'Growing Spheres',
    'FACE',
    'DiCE',
]


def x_cf_columns_for_dim(d: int) -> list[str]:
    return [f'x_cf_{idx}' for idx in range(int(d))]


def load_train_by_dataset(dataset_order: list[str]) -> dict[str, np.ndarray]:
    registries = create_default_registries()
    train_by_dataset = {}
    for dataset_name in dataset_order:
        dataset = registries['dataset'].create(dataset_name, data_dir=str(ROOT / 'data'))
        dataset.load()
        x_train, _ = dataset.get_train()
        train_by_dataset[dataset_name] = np.asarray(x_train, dtype=np.float32)
    return train_by_dataset


def compute_lof_scores(df: pd.DataFrame, dataset_order: list[str]) -> pd.DataFrame:
    train_by_dataset = load_train_by_dataset(dataset_order)
    rows = []

    for dataset_name in dataset_order:
        x_train = train_by_dataset[dataset_name]
        n_neighbors = min(LOF_N_NEIGHBORS, max(1, len(x_train) - 1))
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, novelty=True, n_jobs=-1)
        lof.fit(x_train)

        ds_df = df[df['dataset'].astype(str).eq(dataset_name)].copy()
        cf_cols = x_cf_columns_for_dim(x_train.shape[1])
        for method_label in PAPER_LOF_METHOD_ORDER:
            method_df = ds_df[
                ds_df['method_label'].astype(str).eq(method_label)
                & ds_df['success'].astype(bool)
            ].copy()
            if method_df.empty or not set(cf_cols).issubset(method_df.columns):
                continue

            x_cf = method_df[cf_cols].to_numpy(dtype=np.float32)
            finite_mask = np.isfinite(x_cf).all(axis=1)
            if not finite_mask.any():
                continue

            scored = method_df.loc[finite_mask, ['dataset', 'method_label', 'method', 'run_name', 'query_idx', 'target_class']].copy()
            novelty_score = lof.score_samples(x_cf[finite_mask])
            scored['lof_novelty_score'] = novelty_score
            scored['lof_outlier_score'] = -novelty_score
            scored['lof_log_outlier_score'] = np.log1p(np.maximum(scored['lof_outlier_score'].to_numpy(dtype=float), 0.0))
            scored['lof_n_neighbors'] = int(n_neighbors)
            rows.append(scored)

    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


if LOF_SCORE_PATH.exists() and not RECOMPUTE_LOF:
    LOF_SCORES_DF = pd.read_parquet(LOF_SCORE_PATH)
else:
    LOF_SCORE_PATH.parent.mkdir(parents=True, exist_ok=True)
    LOF_SCORES_DF = compute_lof_scores(ANALYSIS_DF, DATASET_ORDER)
    LOF_SCORES_DF.to_parquet(LOF_SCORE_PATH, index=False, compression='gzip')

if 'lof_log_outlier_score' not in LOF_SCORES_DF.columns:
    LOF_SCORES_DF['lof_log_outlier_score'] = np.log1p(
        np.maximum(LOF_SCORES_DF['lof_outlier_score'].to_numpy(dtype=float), 0.0)
    )

LOF_SCORES_DF = LOF_SCORES_DF[
    LOF_SCORES_DF['method_label'].astype(str).isin(PAPER_LOF_METHOD_ORDER)
].copy()
LOF_SCORES_DF['method_label'] = pd.Categorical(
    LOF_SCORES_DF['method_label'], PAPER_LOF_METHOD_ORDER, ordered=True
)

LOF_BY_DATASET_METHOD_DF = (
    LOF_SCORES_DF
    .groupby(['dataset', 'method_label'], observed=True)
    .agg(
        n_success=('lof_outlier_score', 'size'),
        lof_mean=('lof_outlier_score', 'mean'),
        lof_median=('lof_outlier_score', 'median'),
        log_lof_mean=('lof_log_outlier_score', 'mean'),
        log_lof_median=('lof_log_outlier_score', 'median'),
        lof_std=('lof_outlier_score', 'std'),
        log_lof_std=('lof_log_outlier_score', 'std'),
    )
    .reset_index()
)
LOF_BY_DATASET_METHOD_DF['dataset'] = pd.Categorical(LOF_BY_DATASET_METHOD_DF['dataset'], DATASET_ORDER, ordered=True)
LOF_BY_DATASET_METHOD_DF['method_label'] = pd.Categorical(LOF_BY_DATASET_METHOD_DF['method_label'], PAPER_LOF_METHOD_ORDER, ordered=True)
LOF_BY_DATASET_METHOD_DF = LOF_BY_DATASET_METHOD_DF.sort_values(['dataset', 'method_label'], ignore_index=True)

LOF_AVERAGED_DF = (
    LOF_BY_DATASET_METHOD_DF
    .groupby('method_label', observed=True)
    .agg(
        n_datasets=('dataset', 'nunique'),
        dataset_avg_lof_median=('lof_median', 'mean'),
        dataset_std_lof_median=('lof_median', 'std'),
        dataset_avg_log_lof_mean=('log_lof_mean', 'mean'),
        dataset_std_log_lof_mean=('log_lof_mean', 'std'),
        dataset_avg_log_lof_std=('log_lof_std', 'mean'),
        total_success=('n_success', 'sum'),
    )
    .reset_index()
)
query_weighted = (
    LOF_SCORES_DF
    .groupby('method_label', observed=True)
    .agg(
        query_weighted_log_lof_mean=('lof_log_outlier_score', 'mean'),
        query_weighted_log_lof_std=('lof_log_outlier_score', 'std'),
    )
    .reset_index()
)
LOF_AVERAGED_DF = LOF_AVERAGED_DF.merge(query_weighted, on='method_label', how='left')
LOF_AVERAGED_DF['method_label'] = pd.Categorical(LOF_AVERAGED_DF['method_label'], PAPER_LOF_METHOD_ORDER, ordered=True)
LOF_AVERAGED_DF = LOF_AVERAGED_DF.sort_values('method_label', ignore_index=True)

print({'lof_rows': int(len(LOF_SCORES_DF)), 'cache': str(LOF_SCORE_PATH)})
display(LOF_BY_DATASET_METHOD_DF.round(4))
display(LOF_AVERAGED_DF.round(4))

## Isolation Forest Plausibility

Isolation Forest is fitted once per dataset on the training set and evaluated on successful counterfactuals. We report the outlier score as `-score_samples`, so lower values are less outlier-like. This gives a global anomaly-detection complement to the local LOF metric.

In [ ]:
from sklearn.ensemble import IsolationForest

IFOREST_N_ESTIMATORS = 200
IFOREST_RANDOM_STATE = 0
IFOREST_SCORE_PATH = RESULTS_DIR / 'approved_plot_data' / 'iforest_scores_approved_with_certcf_base.parquet'
RECOMPUTE_IFOREST = False
PAPER_IFOREST_METHOD_ORDER = PAPER_LOF_METHOD_ORDER


def compute_iforest_scores(df: pd.DataFrame, dataset_order: list[str]) -> pd.DataFrame:
    train_by_dataset = load_train_by_dataset(dataset_order)
    rows = []

    for dataset_name in dataset_order:
        x_train = train_by_dataset[dataset_name]
        iforest = IsolationForest(
            n_estimators=IFOREST_N_ESTIMATORS,
            contamination='auto',
            random_state=IFOREST_RANDOM_STATE,
            n_jobs=-1,
        )
        iforest.fit(x_train)

        ds_df = df[df['dataset'].astype(str).eq(dataset_name)].copy()
        cf_cols = x_cf_columns_for_dim(x_train.shape[1])
        for method_label in PAPER_IFOREST_METHOD_ORDER:
            method_df = ds_df[
                ds_df['method_label'].astype(str).eq(method_label)
                & ds_df['success'].astype(bool)
            ].copy()
            if method_df.empty or not set(cf_cols).issubset(method_df.columns):
                continue

            x_cf = method_df[cf_cols].to_numpy(dtype=np.float32)
            finite_mask = np.isfinite(x_cf).all(axis=1)
            if not finite_mask.any():
                continue

            scored = method_df.loc[finite_mask, ['dataset', 'method_label', 'method', 'run_name', 'query_idx', 'target_class']].copy()
            novelty_score = iforest.score_samples(x_cf[finite_mask])
            scored['iforest_novelty_score'] = novelty_score
            scored['iforest_outlier_score'] = -novelty_score
            scored['iforest_n_estimators'] = IFOREST_N_ESTIMATORS
            rows.append(scored)

    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


if IFOREST_SCORE_PATH.exists() and not RECOMPUTE_IFOREST:
    IFOREST_SCORES_DF = pd.read_parquet(IFOREST_SCORE_PATH)
else:
    IFOREST_SCORE_PATH.parent.mkdir(parents=True, exist_ok=True)
    IFOREST_SCORES_DF = compute_iforest_scores(ANALYSIS_DF, DATASET_ORDER)
    IFOREST_SCORES_DF.to_parquet(IFOREST_SCORE_PATH, index=False, compression='gzip')

IFOREST_SCORES_DF = IFOREST_SCORES_DF[
    IFOREST_SCORES_DF['method_label'].astype(str).isin(PAPER_IFOREST_METHOD_ORDER)
].copy()
IFOREST_SCORES_DF['method_label'] = pd.Categorical(
    IFOREST_SCORES_DF['method_label'], PAPER_IFOREST_METHOD_ORDER, ordered=True
)

IFOREST_BY_DATASET_METHOD_DF = (
    IFOREST_SCORES_DF
    .groupby(['dataset', 'method_label'], observed=True)
    .agg(
        n_success=('iforest_outlier_score', 'size'),
        iforest_mean=('iforest_outlier_score', 'mean'),
        iforest_median=('iforest_outlier_score', 'median'),
        iforest_std=('iforest_outlier_score', 'std'),
    )
    .reset_index()
)
IFOREST_BY_DATASET_METHOD_DF['dataset'] = pd.Categorical(IFOREST_BY_DATASET_METHOD_DF['dataset'], DATASET_ORDER, ordered=True)
IFOREST_BY_DATASET_METHOD_DF['method_label'] = pd.Categorical(IFOREST_BY_DATASET_METHOD_DF['method_label'], PAPER_IFOREST_METHOD_ORDER, ordered=True)
IFOREST_BY_DATASET_METHOD_DF = IFOREST_BY_DATASET_METHOD_DF.sort_values(['dataset', 'method_label'], ignore_index=True)

IFOREST_AVERAGED_DF = (
    IFOREST_BY_DATASET_METHOD_DF
    .groupby('method_label', observed=True)
    .agg(
        n_datasets=('dataset', 'nunique'),
        dataset_avg_iforest_mean=('iforest_mean', 'mean'),
        dataset_std_iforest_mean=('iforest_mean', 'std'),
        dataset_avg_iforest_median=('iforest_median', 'mean'),
        dataset_std_iforest_median=('iforest_median', 'std'),
        dataset_avg_iforest_std=('iforest_std', 'mean'),
        total_success=('n_success', 'sum'),
    )
    .reset_index()
)
query_weighted_iforest = (
    IFOREST_SCORES_DF
    .groupby('method_label', observed=True)
    .agg(
        query_weighted_iforest_mean=('iforest_outlier_score', 'mean'),
        query_weighted_iforest_std=('iforest_outlier_score', 'std'),
    )
    .reset_index()
)
IFOREST_AVERAGED_DF = IFOREST_AVERAGED_DF.merge(query_weighted_iforest, on='method_label', how='left')
IFOREST_AVERAGED_DF['method_label'] = pd.Categorical(IFOREST_AVERAGED_DF['method_label'], PAPER_IFOREST_METHOD_ORDER, ordered=True)
IFOREST_AVERAGED_DF = IFOREST_AVERAGED_DF.sort_values('method_label', ignore_index=True)

print({'iforest_rows': int(len(IFOREST_SCORES_DF)), 'cache': str(IFOREST_SCORE_PATH)})
display(IFOREST_BY_DATASET_METHOD_DF.round(4))
display(IFOREST_AVERAGED_DF.round(4))

## Effective Anonymity

Instance-identifiability metric for generated counterfactuals. For each successful counterfactual and requested target class, target-class training points receive Gaussian soft-neighbor weights, and `A_eff = 1 / sum_i w_i^2`. Values near 1 indicate linkage to one dominant training instance; larger values indicate support from several plausible target-class neighbors.

The metric set includes `CertCF alpha=0.25` from `benchmark_full_reduced10000_L1.parquet`, the pre-NN-return CertCF run, alongside the NN-initialized/random-support CertCF variant.

In [ ]:
from counterfactuals.benchmarks import create_default_registries


ANONYMITY_TAU = 1.0
ANONYMITY_CHUNK_SIZE = 128
ANONYMITY_RISK_THRESHOLDS = (2.0, 5.0, 10.0)
ANONYMITY_SCORE_PATH = RESULTS_DIR / 'approved_plot_data' / f'effective_anonymity_approved_tau{str(ANONYMITY_TAU).replace(".", "p")}_with_certcf_base.parquet'
RECOMPUTE_ANONYMITY = False
PAPER_ANONYMITY_METHOD_ORDER = list(PAPER_LOF_METHOD_ORDER) if 'PAPER_LOF_METHOD_ORDER' in globals() else METHOD_ORDER


def anonymity_x_cf_columns_for_dim(d: int) -> list[str]:
    return [f'x_cf_{idx}' for idx in range(int(d))]


def _as_2d_float_array(x: np.ndarray) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[None, :]
    if arr.ndim != 2:
        raise ValueError('Expected a 1-D or 2-D feature array.')
    return arr


def effective_anonymity_batch(
    X_cf: np.ndarray,
    X_train_target: np.ndarray,
    tau: float,
    *,
    chunk_size: int = ANONYMITY_CHUNK_SIZE,
) -> np.ndarray:
    """Compute A_eff for many counterfactuals against target-class train points.

    Uses squared Euclidean distances and a stable shifted softmax equivalent:
    A_eff = (sum_i exp(a_i))^2 / sum_i exp(2 a_i), where a_i = -d_i^2 / tau^2.
    """
    X_cf = _as_2d_float_array(X_cf)
    X_train_target = _as_2d_float_array(X_train_target)
    tau = float(tau)
    if tau <= 0 or not np.isfinite(tau):
        raise ValueError('tau must be a positive finite scalar.')
    if X_train_target.shape[0] == 0:
        return np.full(X_cf.shape[0], np.nan, dtype=np.float64)
    if X_cf.shape[1] != X_train_target.shape[1]:
        raise ValueError(
            f'Feature dimension mismatch: X_cf has {X_cf.shape[1]} columns, '
            f'X_train_target has {X_train_target.shape[1]} columns.'
        )

    out = np.empty(X_cf.shape[0], dtype=np.float64)
    train_sq = np.sum(np.square(X_train_target, dtype=np.float64), axis=1)[None, :]
    tau_sq = tau * tau
    chunk_size = max(1, int(chunk_size))

    for start in range(0, X_cf.shape[0], chunk_size):
        stop = min(start + chunk_size, X_cf.shape[0])
        chunk = X_cf[start:stop]
        chunk_sq = np.sum(np.square(chunk, dtype=np.float64), axis=1)[:, None]
        d2 = chunk_sq + train_sq - 2.0 * np.asarray(chunk, dtype=np.float64) @ np.asarray(X_train_target, dtype=np.float64).T
        np.maximum(d2, 0.0, out=d2)

        logits = -d2 / tau_sq
        logits -= np.max(logits, axis=1, keepdims=True)
        weights_unnormalized = np.exp(logits)
        weight_sum = np.sum(weights_unnormalized, axis=1)
        weight_sq_sum = np.sum(np.square(weights_unnormalized), axis=1)
        out[start:stop] = np.square(weight_sum) / weight_sq_sum

    return out


def effective_anonymity(x_cf: np.ndarray, X_train_target: np.ndarray, tau: float) -> float:
    """Compute A_eff for one counterfactual against target-class train points."""
    return float(effective_anonymity_batch(x_cf, X_train_target, tau, chunk_size=1)[0])


def load_train_with_labels_by_dataset(dataset_order: list[str]) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    registries = create_default_registries()
    train_by_dataset = {}
    for dataset_name in dataset_order:
        dataset = registries['dataset'].create(dataset_name, data_dir=str(ROOT / 'data'))
        dataset.load()
        x_train, y_train = dataset.get_train()
        train_by_dataset[dataset_name] = (
            np.asarray(x_train, dtype=np.float32),
            np.asarray(y_train, dtype=np.int64),
        )
    return train_by_dataset


def compute_effective_anonymity_scores(
    df: pd.DataFrame,
    dataset_order: list[str],
    tau: float,
    *,
    method_order: list[str] = PAPER_ANONYMITY_METHOD_ORDER,
    valid_only: bool = True,
    train_class_by_dataset: dict[str, np.ndarray] | None = None,
    chunk_size: int = ANONYMITY_CHUNK_SIZE,
) -> pd.DataFrame:
    """Score generated CFs with A_eff using labeled or supplied target-class train pools.

    By default the target-class training pool uses dataset labels. Pass
    train_class_by_dataset={dataset_name: predicted_classes} to use model-predicted
    training classes without changing the scoring code.
    """
    train_by_dataset = load_train_with_labels_by_dataset(dataset_order)
    rows = []

    for dataset_name in dataset_order:
        x_train, y_train_label = train_by_dataset[dataset_name]
        y_train_class = (
            np.asarray(train_class_by_dataset[dataset_name], dtype=np.int64)
            if train_class_by_dataset is not None and dataset_name in train_class_by_dataset
            else y_train_label
        )
        ds_df = df[df['dataset'].astype(str).eq(dataset_name)].copy()
        cf_cols = anonymity_x_cf_columns_for_dim(x_train.shape[1])
        if ds_df.empty or not set(cf_cols).issubset(ds_df.columns):
            continue

        for method_label in method_order:
            method_df = ds_df[ds_df['method_label'].astype(str).eq(method_label)].copy()
            if valid_only:
                method_df = method_df[method_df['success'].astype(bool)].copy()
            method_df = method_df[method_df['target_class'].notna()].copy()
            if method_df.empty:
                continue

            for target_class, target_df in method_df.groupby('target_class', observed=True):
                target_class = int(target_class)
                X_train_target = x_train[y_train_class == target_class]
                if len(X_train_target) == 0:
                    continue

                x_cf = target_df[cf_cols].to_numpy(dtype=np.float32)
                finite_mask = np.isfinite(x_cf).all(axis=1)
                if not finite_mask.any():
                    continue

                scored = target_df.loc[
                    finite_mask,
                    ['dataset', 'method_label', 'method', 'run_name', 'query_idx', 'target_class'],
                ].copy()
                scored['effective_anonymity'] = effective_anonymity_batch(
                    x_cf[finite_mask],
                    X_train_target,
                    tau,
                    chunk_size=chunk_size,
                )
                scored['anonymity_tau'] = float(tau)
                scored['target_train_count'] = int(len(X_train_target))
                scored['train_class_source'] = 'provided' if train_class_by_dataset is not None else 'label'
                rows.append(scored)

    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


def summarize_effective_anonymity(
    scores_df: pd.DataFrame,
    *,
    group_cols: list[str],
    risk_thresholds: tuple[float, ...] = ANONYMITY_RISK_THRESHOLDS,
) -> pd.DataFrame:
    if scores_df.empty:
        return pd.DataFrame()

    rows = []
    for keys, group in scores_df.groupby(group_cols, observed=True, sort=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        values = group['effective_anonymity'].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            continue

        row = dict(zip(group_cols, keys))
        row.update({
            'n_success': int(len(values)),
            'aeff_mean': float(np.mean(values)),
            'aeff_median': float(np.median(values)),
            'aeff_std': float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
            'aeff_q25': float(np.quantile(values, 0.25)),
            'aeff_q75': float(np.quantile(values, 0.75)),
        })
        for threshold in risk_thresholds:
            threshold_label = str(threshold).replace('.', 'p').rstrip('0').rstrip('p')
            row[f'aeff_lt_{threshold_label}_rate'] = float(np.mean(values < threshold))
        rows.append(row)

    return pd.DataFrame(rows)


def plot_effective_anonymity_mean(summary_df: pd.DataFrame):
    plot_df = summary_df.copy()
    plot_df['method_label'] = pd.Categorical(plot_df['method_label'], PAPER_ANONYMITY_METHOD_ORDER, ordered=True)
    plot_df = plot_df.sort_values('method_label')

    fig, ax = plt.subplots(figsize=(9.2, 4.8))
    colors = [PALETTE.get(str(label), '#868E96') for label in plot_df['method_label']]
    ax.bar(plot_df['method_label'].astype(str), plot_df['aeff_mean'], color=colors, alpha=0.88)
    ax.set_ylabel('Mean effective anonymity size')
    ax.set_title(f'Instance privacy by method (tau={ANONYMITY_TAU:g})', fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(True, axis='y', alpha=0.22)
    fig.tight_layout()
    return fig


def plot_effective_anonymity_risk(summary_df: pd.DataFrame, threshold: float = 5.0):
    threshold_label = str(float(threshold)).replace('.', 'p').rstrip('0').rstrip('p')
    rate_col = f'aeff_lt_{threshold_label}_rate'
    if rate_col not in summary_df.columns:
        raise KeyError(f'Missing risk-rate column: {rate_col}')

    plot_df = summary_df.copy()
    plot_df['method_label'] = pd.Categorical(plot_df['method_label'], PAPER_ANONYMITY_METHOD_ORDER, ordered=True)
    plot_df = plot_df.sort_values('method_label')

    fig, ax = plt.subplots(figsize=(9.2, 4.8))
    colors = [PALETTE.get(str(label), '#868E96') for label in plot_df['method_label']]
    ax.bar(plot_df['method_label'].astype(str), 100.0 * plot_df[rate_col], color=colors, alpha=0.88)
    ax.set_ylabel(f'CFs with A_eff < {threshold:g} (%)')
    ax.set_title('Overly identifying counterfactual rate by method', fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0.0)
    ax.tick_params(axis='x', rotation=25)
    ax.grid(True, axis='y', alpha=0.22)
    fig.tight_layout()
    return fig


if ANONYMITY_SCORE_PATH.exists() and not RECOMPUTE_ANONYMITY:
    ANONYMITY_SCORES_DF = pd.read_parquet(ANONYMITY_SCORE_PATH)
else:
    ANONYMITY_SCORE_PATH.parent.mkdir(parents=True, exist_ok=True)
    ANONYMITY_SCORES_DF = compute_effective_anonymity_scores(
        ANALYSIS_DF,
        DATASET_ORDER,
        ANONYMITY_TAU,
    )
    ANONYMITY_SCORES_DF.to_parquet(ANONYMITY_SCORE_PATH, index=False, compression='gzip')

ANONYMITY_SCORES_DF = ANONYMITY_SCORES_DF[
    ANONYMITY_SCORES_DF['method_label'].astype(str).isin(PAPER_ANONYMITY_METHOD_ORDER)
].copy()
ANONYMITY_SCORES_DF['method_label'] = pd.Categorical(
    ANONYMITY_SCORES_DF['method_label'], PAPER_ANONYMITY_METHOD_ORDER, ordered=True
)

ANONYMITY_BY_DATASET_METHOD_DF = summarize_effective_anonymity(
    ANONYMITY_SCORES_DF,
    group_cols=['dataset', 'method_label'],
)
ANONYMITY_BY_DATASET_METHOD_DF['dataset'] = pd.Categorical(ANONYMITY_BY_DATASET_METHOD_DF['dataset'], DATASET_ORDER, ordered=True)
ANONYMITY_BY_DATASET_METHOD_DF['method_label'] = pd.Categorical(
    ANONYMITY_BY_DATASET_METHOD_DF['method_label'], PAPER_ANONYMITY_METHOD_ORDER, ordered=True
)
ANONYMITY_BY_DATASET_METHOD_DF = ANONYMITY_BY_DATASET_METHOD_DF.sort_values(['dataset', 'method_label'], ignore_index=True)

ANONYMITY_BY_METHOD_DF = summarize_effective_anonymity(
    ANONYMITY_SCORES_DF,
    group_cols=['method_label'],
)
ANONYMITY_BY_METHOD_DF['method_label'] = pd.Categorical(
    ANONYMITY_BY_METHOD_DF['method_label'], PAPER_ANONYMITY_METHOD_ORDER, ordered=True
)
ANONYMITY_BY_METHOD_DF = ANONYMITY_BY_METHOD_DF.sort_values('method_label', ignore_index=True)

print({'anonymity_rows': int(len(ANONYMITY_SCORES_DF)), 'tau': ANONYMITY_TAU, 'cache': str(ANONYMITY_SCORE_PATH)})
display(ANONYMITY_BY_DATASET_METHOD_DF.round(4))
display(ANONYMITY_BY_METHOD_DF.round(4))
plot_effective_anonymity_mean(ANONYMITY_BY_METHOD_DF);
plot_effective_anonymity_risk(ANONYMITY_BY_METHOD_DF, threshold=5.0);

## Validity vs Privacy

Privacy analogue of the validity/proximity curve. For each effective-anonymity threshold, the curve reports the percentage of all benchmark queries that received a valid counterfactual with `A_eff` at least that threshold. Higher curves are better: they combine validity with lower instance identifiability.

In [ ]:
PRIVACY_AEFF_N_THRESHOLDS = 240
PRIVACY_AEFF_X_MAX = None
PRIVACY_AEFF_MIN_THRESHOLD = 1.0
PRIVACY_CURVE_METHOD_ORDER = PAPER_ANONYMITY_METHOD_ORDER


def privacy_threshold_grid(
    scores_df: pd.DataFrame,
    *,
    n_thresholds: int = PRIVACY_AEFF_N_THRESHOLDS,
    x_max: float | None = PRIVACY_AEFF_X_MAX,
) -> np.ndarray:
    values = scores_df['effective_anonymity'].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.array([PRIVACY_AEFF_MIN_THRESHOLD], dtype=float)
    if x_max is None:
        x_max = float(np.nanquantile(values, 0.95))
        x_max = max(10.0, x_max)
    x_max = max(float(x_max), PRIVACY_AEFF_MIN_THRESHOLD)
    return np.linspace(PRIVACY_AEFF_MIN_THRESHOLD, x_max, int(n_thresholds))


def build_validity_privacy_curve_df(
    scores_df: pd.DataFrame,
    denominator_df: pd.DataFrame,
    *,
    thresholds: np.ndarray | None = None,
    dataset_order: list[str] = DATASET_ORDER,
    method_order: list[str] = PRIVACY_CURVE_METHOD_ORDER,
) -> pd.DataFrame:
    if thresholds is None:
        thresholds = privacy_threshold_grid(scores_df)
    rows = []

    for dataset in dataset_order:
        ds_den = denominator_df[denominator_df['dataset'].astype(str).eq(dataset)].copy()
        ds_scores = scores_df[scores_df['dataset'].astype(str).eq(dataset)].copy()
        if ds_den.empty or ds_scores.empty:
            continue

        for method_label in method_order:
            total = int(ds_den[ds_den['method_label'].astype(str).eq(method_label)].shape[0])
            if total == 0:
                continue
            method_scores = ds_scores[ds_scores['method_label'].astype(str).eq(method_label)].copy()
            values = method_scores['effective_anonymity'].dropna().to_numpy(dtype=float)
            values = values[np.isfinite(values)]
            values_sorted = np.sort(values)
            n_valid_scored = int(len(values_sorted))
            final_validity_pct = 100.0 * n_valid_scored / total

            for threshold in thresholds:
                n_private_valid = n_valid_scored - int(np.searchsorted(values_sorted, threshold, side='left'))
                private_validity_pct = 100.0 * n_private_valid / total
                private_among_valid_pct = 100.0 * n_private_valid / n_valid_scored if n_valid_scored else np.nan
                rows.append({
                    'dataset': dataset,
                    'method_label': method_label,
                    'aeff_threshold': float(threshold),
                    'private_validity_pct': float(private_validity_pct),
                    'private_among_valid_pct': float(private_among_valid_pct),
                    'final_validity_pct': float(final_validity_pct),
                    'n_total': int(total),
                    'n_valid_scored': n_valid_scored,
                })

    return pd.DataFrame(rows)


def aggregate_validity_privacy_curve(curve_df: pd.DataFrame) -> pd.DataFrame:
    if curve_df.empty:
        return pd.DataFrame()
    return (
        curve_df
        .groupby(['method_label', 'aeff_threshold'], observed=True, sort=False)
        .agg(
            mean_private_validity_pct=('private_validity_pct', 'mean'),
            std_private_validity_pct=('private_validity_pct', 'std'),
            mean_private_among_valid_pct=('private_among_valid_pct', 'mean'),
            mean_final_validity_pct=('final_validity_pct', 'mean'),
            n_datasets=('dataset', 'nunique'),
            total_queries=('n_total', 'sum'),
        )
        .reset_index()
        .fillna({'std_private_validity_pct': 0.0})
    )


def plot_validity_vs_privacy_curve(aggregate_curve_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(10.5, 6.0))
    endpoints = []
    for method_label in PRIVACY_CURVE_METHOD_ORDER:
        curve = aggregate_curve_df[aggregate_curve_df['method_label'].astype(str).eq(method_label)].sort_values('aeff_threshold')
        if curve.empty:
            continue
        x = curve['aeff_threshold'].to_numpy(dtype=float)
        y = curve['mean_private_validity_pct'].to_numpy(dtype=float)
        y_std = curve['std_private_validity_pct'].to_numpy(dtype=float)
        color = PALETTE.get(method_label, '#868E96')
        ax.plot(
            x,
            y,
            color=color,
            linewidth=2.7 if method_label.startswith('CertCF') else 2.1,
            alpha=0.96,
            label=method_label,
        )
        ax.fill_between(
            x,
            np.clip(y - y_std, 0.0, 100.0),
            np.clip(y + y_std, 0.0, 100.0),
            color=color,
            alpha=0.10 if method_label.startswith('CertCF') else 0.07,
            linewidth=0,
        )
        endpoints.append({
            'method_label': method_label,
            'y': float(y[-1]),
            'final_y': float(y[-1]),
            'color': color,
        })

    ax.set_xlim(left=PRIVACY_AEFF_MIN_THRESHOLD)
    ax.set_ylim(-2.0, 102.0)
    ax.set_xlabel('Minimum effective anonymity size threshold')
    ax.set_ylabel('Dataset-averaged valid and private CFs (%)')
    ax.set_title('Validity vs instance privacy', fontsize=14, fontweight='bold')
    ax.grid(True, axis='both', alpha=0.22)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    ax.set_xscale('log')
    return fig


VALIDITY_PRIVACY_CURVE_DF = build_validity_privacy_curve_df(
    ANONYMITY_SCORES_DF,
    ANALYSIS_DF,
)
AGGREGATE_VALIDITY_PRIVACY_CURVE_DF = aggregate_validity_privacy_curve(VALIDITY_PRIVACY_CURVE_DF)
AGGREGATE_VALIDITY_PRIVACY_CURVE_DF['method_label'] = pd.Categorical(
    AGGREGATE_VALIDITY_PRIVACY_CURVE_DF['method_label'], PRIVACY_CURVE_METHOD_ORDER, ordered=True
)
AGGREGATE_VALIDITY_PRIVACY_CURVE_DF = AGGREGATE_VALIDITY_PRIVACY_CURVE_DF.sort_values(
    ['method_label', 'aeff_threshold'],
    ignore_index=True,
)

VALIDITY_PRIVACY_AUC_DF = (
    AGGREGATE_VALIDITY_PRIVACY_CURVE_DF
    .groupby('method_label', observed=True, sort=False)
    .apply(
        lambda g: pd.Series({
            'privacy_validity_auc': float(
                (np.trapezoid(
                    g.sort_values('aeff_threshold')['mean_private_validity_pct'].to_numpy(dtype=float) / 100.0,
                    g.sort_values('aeff_threshold')['aeff_threshold'].to_numpy(dtype=float),
                ) if hasattr(np, 'trapezoid') else np.trapz(
                    g.sort_values('aeff_threshold')['mean_private_validity_pct'].to_numpy(dtype=float) / 100.0,
                    g.sort_values('aeff_threshold')['aeff_threshold'].to_numpy(dtype=float),
                ))
                / max(
                    g['aeff_threshold'].max() - g['aeff_threshold'].min(),
                    1e-12,
                )
            ),
            'valid_private_at_aeff_2': float(np.interp(2.0, g.sort_values('aeff_threshold')['aeff_threshold'], g.sort_values('aeff_threshold')['mean_private_validity_pct'])),
            'valid_private_at_aeff_5': float(np.interp(5.0, g.sort_values('aeff_threshold')['aeff_threshold'], g.sort_values('aeff_threshold')['mean_private_validity_pct'])),
            'valid_private_at_aeff_10': float(np.interp(10.0, g.sort_values('aeff_threshold')['aeff_threshold'], g.sort_values('aeff_threshold')['mean_private_validity_pct'])),
        })
    )
    .reset_index()
)
VALIDITY_PRIVACY_AUC_DF['method_label'] = pd.Categorical(VALIDITY_PRIVACY_AUC_DF['method_label'], PRIVACY_CURVE_METHOD_ORDER, ordered=True)
VALIDITY_PRIVACY_AUC_DF = VALIDITY_PRIVACY_AUC_DF.sort_values('method_label', ignore_index=True)

display(VALIDITY_PRIVACY_AUC_DF.round(4))
plot_validity_vs_privacy_curve(AGGREGATE_VALIDITY_PRIVACY_CURVE_DF);

## Privacy vs Manifoldness

Joint analysis of instance privacy and LOF plausibility. The curve asks: among counterfactuals that are no more outlier-like than a dataset-specific LOF quantile, how large is their effective anonymity? Higher `A_eff` and lower admitted LOF quantiles indicate a better private-and-plausible tradeoff.

In [ ]:
PRIVACY_MANIFOLD_METHOD_ORDER = PAPER_ANONYMITY_METHOD_ORDER
PRIVACY_MANIFOLD_QUANTILES = np.linspace(0.05, 1.0, 20)


def build_privacy_manifold_joint_df(
    anonymity_df: pd.DataFrame,
    lof_df: pd.DataFrame,
) -> pd.DataFrame:
    keys = ['dataset', 'method_label', 'query_idx', 'target_class']
    left = anonymity_df[keys + ['effective_anonymity', 'anonymity_tau']].copy()
    right = lof_df[keys + ['lof_outlier_score', 'lof_log_outlier_score']].copy()
    joint = left.merge(right, on=keys, how='inner')
    joint = joint[
        joint['effective_anonymity'].notna()
        & joint['lof_outlier_score'].notna()
        & np.isfinite(joint['effective_anonymity'].astype(float))
        & np.isfinite(joint['lof_outlier_score'].astype(float))
    ].copy()
    joint['method_label'] = pd.Categorical(joint['method_label'], PRIVACY_MANIFOLD_METHOD_ORDER, ordered=True)
    return joint.sort_values(['dataset', 'method_label', 'query_idx'], ignore_index=True)


def build_privacy_manifoldness_curve_df(
    joint_df: pd.DataFrame,
    *,
    quantiles: np.ndarray = PRIVACY_MANIFOLD_QUANTILES,
    dataset_order: list[str] = DATASET_ORDER,
    method_order: list[str] = PRIVACY_MANIFOLD_METHOD_ORDER,
) -> pd.DataFrame:
    rows = []
    for dataset in dataset_order:
        ds_joint = joint_df[joint_df['dataset'].astype(str).eq(dataset)].copy()
        if ds_joint.empty:
            continue
        lof_values = ds_joint['lof_outlier_score'].to_numpy(dtype=float)
        lof_values = lof_values[np.isfinite(lof_values)]
        if len(lof_values) == 0:
            continue

        lof_thresholds = np.quantile(lof_values, quantiles)
        for method_label in method_order:
            method_df = ds_joint[ds_joint['method_label'].astype(str).eq(method_label)].copy()
            if method_df.empty:
                continue
            method_lof = method_df['lof_outlier_score'].to_numpy(dtype=float)
            method_aeff = method_df['effective_anonymity'].to_numpy(dtype=float)
            total_success = int(len(method_df))

            for quantile, lof_threshold in zip(quantiles, lof_thresholds):
                mask = method_lof <= float(lof_threshold)
                aeff_values = method_aeff[mask]
                aeff_values = aeff_values[np.isfinite(aeff_values)]
                rows.append({
                    'dataset': dataset,
                    'method_label': method_label,
                    'lof_quantile': float(quantile),
                    'lof_threshold': float(lof_threshold),
                    'n_success': total_success,
                    'n_plausible': int(len(aeff_values)),
                    'plausible_success_pct': float(100.0 * len(aeff_values) / total_success) if total_success else np.nan,
                    'aeff_mean_among_plausible': float(np.mean(aeff_values)) if len(aeff_values) else np.nan,
                    'aeff_median_among_plausible': float(np.median(aeff_values)) if len(aeff_values) else np.nan,
                })

    return pd.DataFrame(rows)


def aggregate_privacy_manifoldness_curve(curve_df: pd.DataFrame) -> pd.DataFrame:
    if curve_df.empty:
        return pd.DataFrame()
    return (
        curve_df
        .groupby(['method_label', 'lof_quantile'], observed=True, sort=False)
        .agg(
            dataset_avg_aeff_mean=('aeff_mean_among_plausible', 'mean'),
            dataset_avg_aeff_median=('aeff_median_among_plausible', 'mean'),
            dataset_std_aeff_median=('aeff_median_among_plausible', 'std'),
            dataset_avg_plausible_success_pct=('plausible_success_pct', 'mean'),
            n_datasets=('dataset', 'nunique'),
            total_plausible=('n_plausible', 'sum'),
            total_success=('n_success', 'sum'),
        )
        .reset_index()
        .fillna({'dataset_std_aeff_median': 0.0})
    )


def privacy_manifoldness_summary(joint_df: pd.DataFrame) -> pd.DataFrame:
    by_dataset = (
        joint_df
        .groupby(['dataset', 'method_label'], observed=True)
        .agg(
            n_success=('effective_anonymity', 'size'),
            aeff_mean=('effective_anonymity', 'mean'),
            aeff_median=('effective_anonymity', 'median'),
            lof_mean=('lof_outlier_score', 'mean'),
            lof_median=('lof_outlier_score', 'median'),
            log_lof_mean=('lof_log_outlier_score', 'mean'),
        )
        .reset_index()
    )
    by_dataset['privacy_risk_median_aeff'] = 1.0 / np.maximum(by_dataset['aeff_median'].astype(float), 1e-12)
    summary = (
        by_dataset
        .groupby('method_label', observed=True)
        .agg(
            n_datasets=('dataset', 'nunique'),
            dataset_avg_aeff_mean=('aeff_mean', 'mean'),
            dataset_avg_aeff_median=('aeff_median', 'mean'),
            dataset_avg_privacy_risk=('privacy_risk_median_aeff', 'mean'),
            dataset_avg_lof_mean=('lof_mean', 'mean'),
            dataset_avg_lof_median=('lof_median', 'mean'),
            dataset_avg_log_lof_mean=('log_lof_mean', 'mean'),
            total_success=('n_success', 'sum'),
        )
        .reset_index()
    )
    summary['method_label'] = pd.Categorical(summary['method_label'], PRIVACY_MANIFOLD_METHOD_ORDER, ordered=True)
    return summary.sort_values('method_label', ignore_index=True)


def plot_privacy_vs_manifoldness_curve(aggregate_curve_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(10.2, 5.8))
    for method_label in PRIVACY_MANIFOLD_METHOD_ORDER:
        curve = aggregate_curve_df[aggregate_curve_df['method_label'].astype(str).eq(method_label)].sort_values('lof_quantile')
        if curve.empty:
            continue
        color = PALETTE.get(method_label, '#868E96')
        ax.plot(
            curve['lof_quantile'],
            curve['dataset_avg_aeff_median'],
            marker='o',
            linewidth=2.7 if method_label.startswith('CertCF') else 2.1,
            color=color,
            alpha=0.95,
            label=method_label,
        )
    ax.set_xlabel('Allowed LOF outlierness quantile within dataset')
    ax.set_ylabel('Dataset-averaged median A_eff among plausible CFs')
    ax.set_title('Privacy vs manifold plausibility', fontsize=14, fontweight='bold')
    ax.grid(True, axis='both', alpha=0.22)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    return fig


def pareto_frontier_lower_left(points_df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    finite = points_df[[x_col, y_col]].notna().all(axis=1)
    frontier = []
    best_y = np.inf
    for _, row in points_df[finite].sort_values([x_col, y_col], ascending=[True, True]).iterrows():
        y = float(row[y_col])
        if y < best_y:
            frontier.append(row)
            best_y = y
    return pd.DataFrame(frontier)


def plot_privacy_manifoldness_scatter(summary_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(7.6, 5.8))
    plot_df = summary_df.copy()
    x_raw_col = 'dataset_avg_log_lof_mean'
    y_raw_col = 'dataset_avg_privacy_risk'
    x_var = float(np.nanvar(plot_df[x_raw_col].to_numpy(dtype=float)))
    y_var = float(np.nanvar(plot_df[y_raw_col].to_numpy(dtype=float)))
    if not np.isfinite(x_var) or x_var <= 0.0:
        x_var = 1.0
    if not np.isfinite(y_var) or y_var <= 0.0:
        y_var = 1.0
    plot_df['variance_scaled_log_lof'] = plot_df[x_raw_col].astype(float) / x_var
    plot_df['variance_scaled_privacy_risk'] = plot_df[y_raw_col].astype(float) / y_var
    frontier = pareto_frontier_lower_left(
        plot_df,
        x_col='variance_scaled_log_lof',
        y_col='variance_scaled_privacy_risk',
    )
    if len(frontier) >= 2:
        ax.plot(
            frontier['variance_scaled_log_lof'],
            frontier['variance_scaled_privacy_risk'],
            color='#222222',
            linewidth=1.6,
            linestyle='--',
            alpha=0.70,
            zorder=1,
        )
    for _, row in plot_df.iterrows():
        method_label = str(row['method_label'])
        color = PALETTE.get(method_label, '#868E96')
        ax.scatter(
            row['variance_scaled_log_lof'],
            row['variance_scaled_privacy_risk'],
            s=95 if method_label.startswith('CertCF') else 75,
            color=color,
            alpha=0.90,
            zorder=2,
            label=method_label,
        )
        ax.annotate(
            SHORT_METHOD_LABELS.get(method_label, method_label),
            xy=(row['variance_scaled_log_lof'], row['variance_scaled_privacy_risk']),
            xytext=(5, 4),
            textcoords='offset points',
            fontsize=9,
            color=color,
        )
    ax.set_xlabel('Variance-scaled log-LOF outlier score (lower is better)')
    ax.set_ylabel('Variance-scaled privacy risk, 1 / median A_eff (lower is better)')
    ax.set_title('Privacy-manifoldness method map', fontsize=14, fontweight='bold')
    ax.grid(True, axis='both', alpha=0.22)
    fig.tight_layout()
    return fig


PRIVACY_MANIFOLD_JOINT_DF = build_privacy_manifold_joint_df(ANONYMITY_SCORES_DF, LOF_SCORES_DF)
PRIVACY_MANIFOLD_CURVE_DF = build_privacy_manifoldness_curve_df(PRIVACY_MANIFOLD_JOINT_DF)
AGGREGATE_PRIVACY_MANIFOLD_CURVE_DF = aggregate_privacy_manifoldness_curve(PRIVACY_MANIFOLD_CURVE_DF)
AGGREGATE_PRIVACY_MANIFOLD_CURVE_DF['method_label'] = pd.Categorical(
    AGGREGATE_PRIVACY_MANIFOLD_CURVE_DF['method_label'], PRIVACY_MANIFOLD_METHOD_ORDER, ordered=True
)
AGGREGATE_PRIVACY_MANIFOLD_CURVE_DF = AGGREGATE_PRIVACY_MANIFOLD_CURVE_DF.sort_values(
    ['method_label', 'lof_quantile'],
    ignore_index=True,
)
PRIVACY_MANIFOLD_SUMMARY_DF = privacy_manifoldness_summary(PRIVACY_MANIFOLD_JOINT_DF)

print({'privacy_manifold_rows': int(len(PRIVACY_MANIFOLD_JOINT_DF))})
display(PRIVACY_MANIFOLD_SUMMARY_DF.round(4))
plot_privacy_vs_manifoldness_curve(AGGREGATE_PRIVACY_MANIFOLD_CURVE_DF);
plot_privacy_manifoldness_scatter(PRIVACY_MANIFOLD_SUMMARY_DF);

## Validity Variability vs Proximity


In [ ]:
def plot_validity_variability_vs_proximity(aggregate_curve_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(10.5, 5.4))

    for method_label in METHOD_ORDER:
        curve = aggregate_curve_df[aggregate_curve_df['method_label'].eq(method_label)].sort_values('normalized_l1_threshold')
        if curve.empty:
            continue

        x = curve['normalized_l1_threshold'].to_numpy(dtype=float)
        y = curve['std_validity_pct'].to_numpy(dtype=float)
        method = str(curve['method'].iloc[0])
        color = PALETTE.get(method_label, '#868E96')
        ax.step(
            x,
            y,
            where='post',
            color=color,
            linewidth=2.5 if method == 'certcf' else 2.0,
            alpha=0.96 if method == 'certcf' else 0.84,
            label=method_label,
        )

    ax.set_xlim(0.0, NORMALIZED_L1_X_MAX)
    ax.set_ylim(bottom=0.0)
    ax.set_xlabel('Normalized L1 proximity threshold (L1 / dataset NN mean L1)')
    ax.set_ylabel('Validity standard deviation across datasets (percentage points)')
    ax.set_title('Validity variability vs normalized L1 proximity', fontsize=14, fontweight='bold')
    ax.grid(True, axis='both', alpha=0.22)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    return fig


plot_validity_variability_vs_proximity(AGGREGATE_NORMALIZED_L1_CURVE_DF);


## Decision Tree Surrogate

A shallow decision tree is trained on the generated counterfactuals of each method. We report two label choices: the requested `target_class` on valid CFs, and the classifier prediction `y_cf` on all generated CFs with a prediction. Higher cross-validated accuracy or macro-F1 with a small tree suggests a simpler, more readable CF structure.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

TREE_MAX_DEPTH = 3
TREE_RANDOM_STATE = 0
TREE_METHOD_ORDER = PAPER_LOF_METHOD_ORDER
TREE_LABEL_CONFIGS = [
    ('target_class_valid_only', 'target_class', 'success'),
    ('classifier_prediction', 'y_cf', None),
]


def available_cf_columns(df: pd.DataFrame) -> list[str]:
    cols = sorted(
        [c for c in df.columns if c.startswith('x_cf_')],
        key=lambda c: int(c.rsplit('_', 1)[1]),
    )
    return [c for c in cols if df[c].notna().all()]


def tree_scores_for_group(group: pd.DataFrame, label_col: str) -> dict | None:
    cf_cols = available_cf_columns(group)
    x = group[cf_cols].to_numpy(dtype=np.float32)
    y = group[label_col].astype(int).to_numpy()
    finite = np.isfinite(x).all(axis=1)
    x, y = x[finite], y[finite]
    class_counts = pd.Series(y).value_counts()
    if len(y) < 10 or len(class_counts) < 2 or class_counts.min() < 2:
        return None

    tree = DecisionTreeClassifier(
        max_depth=TREE_MAX_DEPTH,
        min_samples_leaf=max(2, int(0.02 * len(y))),
        random_state=TREE_RANDOM_STATE,
    )
    cv = StratifiedKFold(
        n_splits=min(5, int(class_counts.min())),
        shuffle=True,
        random_state=TREE_RANDOM_STATE,
    )
    accuracy_scores = cross_val_score(tree, x, y, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(tree, x, y, cv=cv, scoring='f1_macro')
    tree.fit(x, y)

    return {
        'n_samples': int(len(y)),
        'n_classes': int(len(class_counts)),
        'tree_cv_accuracy': float(accuracy_scores.mean()),
        'tree_cv_accuracy_std': float(accuracy_scores.std()),
        'tree_cv_f1_macro': float(f1_scores.mean()),
        'tree_cv_f1_macro_std': float(f1_scores.std()),
        'majority_baseline': float(class_counts.max() / class_counts.sum()),
        'tree_depth': int(tree.get_depth()),
        'tree_leaves': int(tree.get_n_leaves()),
    }


def decision_tree_surrogate_results(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset in DATASET_ORDER:
        ds_df = df[df['dataset'].astype(str).eq(dataset)].copy()
        for method_label in TREE_METHOD_ORDER:
            method_df = ds_df[ds_df['method_label'].astype(str).eq(method_label)].copy()
            for label_source, label_col, mask_col in TREE_LABEL_CONFIGS:
                group = method_df[method_df[label_col].notna()].copy()
                if mask_col in group.columns:
                    group = group[group[mask_col].astype(bool)].copy()
                scores = tree_scores_for_group(group, label_col)
                if scores is None:
                    continue
                rows.append({
                    'dataset': dataset,
                    'method_label': method_label,
                    'label_source': label_source,
                    **scores,
                    'accuracy_gain': scores['tree_cv_accuracy'] - scores['majority_baseline'],
                })
    return pd.DataFrame(rows)


TREE_SURROGATE_DF = decision_tree_surrogate_results(ANALYSIS_DF)
TREE_SURROGATE_DF['dataset'] = pd.Categorical(TREE_SURROGATE_DF['dataset'], DATASET_ORDER, ordered=True)
TREE_SURROGATE_DF['method_label'] = pd.Categorical(TREE_SURROGATE_DF['method_label'], TREE_METHOD_ORDER, ordered=True)
TREE_SURROGATE_DF = TREE_SURROGATE_DF.sort_values(['label_source', 'dataset', 'method_label'], ignore_index=True)

TREE_SURROGATE_AVERAGED_DF = (
    TREE_SURROGATE_DF
    .groupby(['label_source', 'method_label'], observed=True)
    .agg(
        n_datasets=('dataset', 'nunique'),
        avg_tree_cv_accuracy=('tree_cv_accuracy', 'mean'),
        std_tree_cv_accuracy=('tree_cv_accuracy', 'std'),
        avg_tree_cv_f1_macro=('tree_cv_f1_macro', 'mean'),
        std_tree_cv_f1_macro=('tree_cv_f1_macro', 'std'),
        avg_majority_baseline=('majority_baseline', 'mean'),
        avg_accuracy_gain=('accuracy_gain', 'mean'),
        avg_tree_depth=('tree_depth', 'mean'),
        avg_tree_leaves=('tree_leaves', 'mean'),
        total_samples=('n_samples', 'sum'),
    )
    .reset_index()
)
TREE_SURROGATE_AVERAGED_DF['method_label'] = pd.Categorical(
    TREE_SURROGATE_AVERAGED_DF['method_label'], TREE_METHOD_ORDER, ordered=True
)
TREE_SURROGATE_AVERAGED_DF = TREE_SURROGATE_AVERAGED_DF.sort_values(
    ['label_source', 'method_label'],
    ignore_index=True,
)

display(TREE_SURROGATE_DF.round(4))
display(TREE_SURROGATE_AVERAGED_DF.round(4))